# Экспертиза записей Chatterbox на слух: Паоло (it, ru) и Хава (he), gemini-3.1-flash-live-preview

Вход — вывод ноутбука `italo-tutor-chatterbox-eval` (подключён как Input). Ключ Gemini — Kaggle Secret `GEMINI_API_KEY`
(Add-ons → Secrets, галочка для этого ноутбука). GPU не нужен, Internet = On.
Вопросы слепые: эксперт выбирает из вариантов в случайном порядке, что прозвучало; ответ сравнивается с задуманным.

In [ ]:
!pip install -q websockets soundfile
!rm -rf /kaggle/working/task && git clone -q --depth 1 https://github.com/vasiliad/italo-tutor-chatterbox /kaggle/working/task

In [ ]:
import os, glob, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient
key = UserSecretsClient().get_secret("GEMINI_API_KEY")
os.makedirs(os.path.expanduser("~/key"), exist_ok=True)
open(os.path.expanduser("~/key/key"), "w").write(key)

src = glob.glob("/kaggle/input/**/results.csv", recursive=True)
print("найдено:", src)
assert src, "Нет входа: подключите вывод ноутбука italo-tutor-chatterbox-eval (Add Input → Notebooks)"
ROOT = "/kaggle/working/chatterbox_eval"
shutil.copytree(os.path.dirname(src[0]), ROOT, dirs_exist_ok=True)

In [ ]:
# Сначала проверка связи: 3 задания
!cd /kaggle/working/task/tools/expert && python3 expert_listen.py --root /kaggle/working/chatterbox_eval --limit 3 --jobs 1

In [ ]:
# Полная экспертиза
!cd /kaggle/working/task/tools/expert && python3 expert_listen.py --root /kaggle/working/chatterbox_eval --jobs 3

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/working/chatterbox_eval/expert_results.csv")
print(open("/kaggle/working/chatterbox_eval/expert_summary.md").read())
out = "/kaggle/working/expert_out"
os.makedirs(out, exist_ok=True)
for f in ["expert_results.csv", "expert_summary.md"]:
    shutil.copy(f"/kaggle/working/chatterbox_eval/{f}", out)
shutil.rmtree("/kaggle/working/chatterbox_eval"); shutil.rmtree("/kaggle/working/task", ignore_errors=True)
os.remove(os.path.expanduser("~/key/key"))